In [1]:
import argparse, time, os
import random
from tqdm import tqdm
import copy, math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from pyproj import Proj, Transformer, Geod
import xarray
import xrspatial.slope
from bokeh.models import PrintfTickFormatter, FixedTicker
import datashader as ds
import holoviews as hv
from holoviews import opts
from holoviews.operation.datashader import datashade, rasterize, inspect_points
import hvplot.xarray
import hvplot.pandas
import rioxarray
from rasterio.enums import Resampling

hv.extension('bokeh')

In [2]:
bedmap_folder = "/media/maffe/sturellone/gprclean/"
bedmap_file = "bedmap_track_ids.parquet"

In [3]:
t0 = time.time()
bedmap = pd.read_parquet(bedmap_folder+bedmap_file, engine='fastparquet')
print(f'Parquet loaded in {time.time()-t0:.1f}')

Parquet loaded in 7.3


In [4]:
bedmap.head(5)

,flight_id,point_id,lon,lat,date,time_utc,ice_thickness,bed_evel,atd,file,file_no,year_start,year_end,datetime,timestamp,east,north,track_id,track_method
0,-9999,-9999,13.9607,-71.3070,-9999,-9999,625.85,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,1,2007,2008,2007-01-01,1167609600,494218.704757,1.988011e+06,0,spatial
1,-9999,-9999,13.9666,-71.3082,-9999,-9999,602.80,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,1,2007,2008,2007-01-01,1167609600,494391.129111,1.987830e+06,0,spatial
2,-9999,-9999,13.9709,-71.3092,-9999,-9999,602.20,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,1,2007,2008,2007-01-01,1167609600,494513.398572,1.987685e+06,0,spatial
3,-9999,-9999,13.9738,-71.3098,-9999,-9999,604.50,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,1,2007,2008,2007-01-01,1167609600,494597.851986,1.987595e+06,0,spatial
4,-9999,-9999,13.9753,-71.3101,-9999,-9999,575.40,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,1,2007,2008,2007-01-01,1167609600,494641.810228,1.987550e+06,0,spatial


In [5]:
FOLDER_AN_VELOCITY = f"/media/maffe/nvme/Antarctica_NSIDC/velocity/NSIDC-0754/"
FOLDER_AN_THICKNESS = f"/media/maffe/nvme/Antarctica_NSIDC/thickness/NSIDC-0756/"
FOLDER_AN_RACMO = f"/media/maffe/nvme/racmo/antarctica_racmo2.3p2/2km/"
FOLDER_AN_TEMP = f"/media/maffe/nvme/ERA5/"

file_an_v = f"{FOLDER_AN_VELOCITY}antarctic_ice_vel_phase_map_v01.nc"
file_an_smb = f"{FOLDER_AN_RACMO}smb_antarctica_mean_1979_2021_RACMO23p2_gf.nc"
file_an_temp = f"{FOLDER_AN_TEMP}era5land_era5.nc"
file_an_bedmac = f"{FOLDER_AN_THICKNESS}BedMachineAntarctica-v3.nc"
#TODO: replace with bedmachine v4.1
#TODO: add dH/dt

In [6]:
# Open Antarctica arrays
an_v = rioxarray.open_rasterio(file_an_v)
an_smb = rioxarray.open_rasterio(file_an_smb)
an_bedmac = rioxarray.open_rasterio(file_an_bedmac)
an_temp = rioxarray.open_rasterio(file_an_temp)

# Pre-process t2m map
an_temp = an_temp.squeeze()
an_temp = an_temp.sel(y=an_temp.y < -60)
an_temp = an_temp.rio.reproject("EPSG:3031",resampling=Resampling.bilinear)

print(an_v['VX'].rio.resolution())
print(an_v['VY'].rio.resolution())
print(an_bedmac['thickness'].rio.resolution())
print(an_bedmac['surface'].rio.resolution())
print(an_smb.rio.resolution())
print(an_temp.rio.resolution())

an_bedmac['thickness'] = an_bedmac['thickness'].where(an_bedmac['thickness'] != 0.0)

# Calculate the slope
res_elevation = an_bedmac['surface'].rio.resolution()[0] # 500 m
#an_bedmac['slope'] = xrspatial.slope(an_bedmac['surface'].squeeze())
dz_dy, dz_dx = np.gradient(an_bedmac['surface'].squeeze(), res_elevation, res_elevation)
slope_raw = np.sqrt(dz_dx**2 + dz_dy**2)
an_bedmac['slope'] = (('y', 'x'), slope_raw)

nodata_v = an_v['VX'].rio.nodata

an_vx = an_v['VX']
an_vy = an_v['VY']

an_vx.values = np.where(an_vx.values == nodata_v, np.nan, an_vx.values)
an_vy.values = np.where(an_vy.values == nodata_v, np.nan, an_vy.values)

an_v['V'] = (an_vx**2 + an_vy**2)**0.5

assert an_vx.rio.crs == an_vy.rio.crs == an_smb.rio.crs == an_bedmac.rio.crs == an_temp.rio.crs, 'Different crs.'

(450.0, -450.0)
(450.0, -450.0)
(500.0, -500.0)
(500.0, -500.0)
(2000.0, -2000.0)
(2605.1644663456045, -2605.1644663456045)


In [7]:
# Interpolate arrays linearly
#todo: possible improvement: xESMF library with method='conserve'

eastings_ar = xarray.DataArray(bedmap['east'])
northings_ar = xarray.DataArray(bedmap['north'])

elev_interp_data = an_bedmac['surface'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
ith_interp_data = an_bedmac['thickness'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
slope_interp_data = an_bedmac['slope'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
vx_interp_data = an_v['VX'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
vy_interp_data = an_v['VY'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
v_interp_data = an_v['V'].interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
smb_interp_data = an_smb.interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()
temp_interp_data = an_temp.interp(y=northings_ar, x=eastings_ar, method='linear').data.squeeze()

In [8]:
# Plug in dataset

#bedmap['v'] = v_interp_data
bedmap['vx'] = vx_interp_data
bedmap['vy'] = vy_interp_data
bedmap['v'] = np.sqrt(vx_interp_data**2 + vy_interp_data**2)
bedmap['ith_bm'] = ith_interp_data
bedmap['smb'] = smb_interp_data
bedmap['z'] = elev_interp_data
bedmap['s'] = slope_interp_data
bedmap['temp'] = temp_interp_data

In [9]:
print(f"Before dropping nans: {len(bedmap)} points")
bedmap_nonans = bedmap.dropna()
print(f"After dropping nans: {len(bedmap_nonans)} points")

Before dropping nans: 74747031 points
After dropping nans: 73576797 points


In [10]:
bedmap.head(5)

,flight_id,point_id,lon,lat,date,time_utc,ice_thickness,bed_evel,atd,file,...,track_id,track_method,vx,vy,v,ith_bm,smb,z,s,temp
0,-9999,-9999,13.9607,-71.3070,-9999,-9999,625.85,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,...,0,spatial,-7.159775,16.247737,17.755319,721.832218,178.533281,1148.460946,0.020026,253.651290
1,-9999,-9999,13.9666,-71.3082,-9999,-9999,602.80,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,...,0,spatial,-7.950166,16.239861,18.081433,686.471526,179.367294,1152.524994,0.016372,253.629406
2,-9999,-9999,13.9709,-71.3092,-9999,-9999,602.20,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,...,0,spatial,-8.492812,16.226179,18.314387,658.551797,180.061604,1154.592650,0.014020,253.611186
3,-9999,-9999,13.9738,-71.3098,-9999,-9999,604.50,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,...,0,spatial,-8.910468,16.261161,18.542433,643.769784,180.481357,1155.652343,0.014609,253.600447
4,-9999,-9999,13.9753,-71.3101,-9999,-9999,575.40,-9999.0,-9999.0,AWI_2007_ANTR_AIR_BM2.csv,...,0,spatial,-9.127859,16.289290,18.672407,636.126988,180.691821,1156.242109,0.015393,253.595122


In [11]:
# Save
save = True
if save:
    bedmap.to_parquet(f"{bedmap_folder}/bedmap_track_ids_with_climate.parquet", engine='pyarrow', index=False)
    print(f"BedMap dataframe saved as parquet: {len(bedmap)} points.")

BedMap dataframe saved as parquet: 74747031 points.


In [12]:
# plot

points_bedmap = hv.Points(bedmap, ['east', 'north'], 'temp') # HoloViews Points object

formatter = PrintfTickFormatter(format='%i')

# Rasterize the points
rasterized_bedmap = rasterize(points_bedmap,aggregator=ds.mean('temp')).opts(
    cmap=plt.colormaps['turbo'],
    cnorm='eq_hist',
    width=700,
    height=700,
    colorbar=True,
    #xaxis=None,            # Remove x-axis
    #yaxis=None,            # Remove y-axis
    #colorbar_opts={
    #    'title': 'Ice Thickness measurements [m]',
    #    'formatter': formatter,  # Apply custom formatter to the colorbar,
    #    "ticker": FixedTicker(ticks=[500, 1000, 1500, 2000, 2500, 3000, 4500])
    #},
    colorbar_position='bottom',
)


# Display the plot
hv.output(rasterized_bedmap)

:DynamicMap   []
   :Image   [east,north]   (east_north temp)